# 01 — H100 Video Profiling

**Goal:** Measure inference performance on an H100 GPU.

Metrics per second:
- GPU utilization %
- GPU memory usage (MB)
- Inference FPS
- Frame latency (ms)

## 1 — Setup

In [ ]:
!pip install -q torch torchvision pynvml opencv-python matplotlib pyyaml scipy

In [ ]:
import os, sys, time, csv, threading
import cv2
import numpy as np
import torch
from google.colab import drive
drive.mount('/content/drive')

ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
VIDEO_DIR = os.path.join(ECOCAR_ROOT, "video")
WEIGHTS_DIR = os.path.join(ECOCAR_ROOT, "weights")
OUTPUT_DIR = "/content/gpu_profile"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROJECT_DIR = os.path.join(ECOCAR_ROOT, "DETR_GeoLane_pipeline")
if not os.path.isdir(PROJECT_DIR):
    !git clone https://github.com/ChenSiyun1234/EcoCAR-Perception-Pipeline-YOLO26-BDD100K.git /content/repo 2>/dev/null || true
    PROJECT_DIR = "/content/repo/DETR_GeoLane_pipeline"
sys.path.insert(0, PROJECT_DIR)

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

## 2 — NVML GPU Monitor

In [ ]:
import pynvml

pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

class GPUMonitor:
    def __init__(self, interval_ms=100):
        self.interval = interval_ms / 1000.0
        self.samples = []
        self._running = False

    def start(self):
        self._running = True
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        self._thread.join(timeout=2.0)

    def _poll(self):
        while self._running:
            try:
                util = pynvml.nvmlDeviceGetUtilizationRates(handle)
                mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
                self.samples.append({
                    'timestamp': time.time(),
                    'gpu_util': util.gpu,
                    'mem_used_mb': mem.used / (1024**2),
                })
            except Exception:
                pass
            time.sleep(self.interval)

    def per_second_report(self, t0):
        report = []
        if not self.samples:
            return report
        bucket, sec = [], 0
        for s in self.samples:
            t = int(s['timestamp'] - t0)
            if t < 0:
                continue
            if t != sec:
                if bucket:
                    report.append({
                        'second': sec,
                        'gpu_util_avg': np.mean([b['gpu_util'] for b in bucket]),
                        'gpu_util_max': max(b['gpu_util'] for b in bucket),
                        'mem_used_avg_mb': np.mean([b['mem_used_mb'] for b in bucket]),
                    })
                sec, bucket = t, []
            bucket.append(s)
        if bucket:
            report.append({
                'second': sec,
                'gpu_util_avg': np.mean([b['gpu_util'] for b in bucket]),
                'gpu_util_max': max(b['gpu_util'] for b in bucket),
                'mem_used_avg_mb': np.mean([b['mem_used_mb'] for b in bucket]),
            })
        return report

print("GPUMonitor ready.")

## 3 — Load Model

In [ ]:
from src.config import Config
from src.model import build_model

CKPT_PATH = os.path.join(WEIGHTS_DIR, "dualpath_v1_best.pt")
if not os.path.isfile(CKPT_PATH):
    # Try last
    CKPT_PATH = os.path.join(WEIGHTS_DIR, "dualpath_v1_last.pt")

assert os.path.isfile(CKPT_PATH), f"No checkpoint found at {CKPT_PATH}"

ckpt = torch.load(CKPT_PATH, map_location="cuda", weights_only=False)
saved_cfg = ckpt.get("config", {})
cfg = Config.from_dict(saved_cfg) if saved_cfg else Config()

model = build_model(cfg)
model.load_state_dict(ckpt["model_state_dict"], strict=False)
model = model.to("cuda").eval()
print(f"Model loaded from epoch {ckpt.get('epoch', '?')}")

## 4 — Find Video

In [ ]:
video_files = sorted([
    os.path.join(VIDEO_DIR, f) for f in os.listdir(VIDEO_DIR)
    if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))
]) if os.path.isdir(VIDEO_DIR) else []

assert video_files, f"No videos in {VIDEO_DIR}"
INPUT_VIDEO = video_files[0]

cap = cv2.VideoCapture(INPUT_VIDEO)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_vid = cap.get(cv2.CAP_PROP_FPS)
print(f"Video: {os.path.basename(INPUT_VIDEO)} ({total_frames} frames @ {fps_vid:.0f} FPS)")

## 5 — Profiled Inference

In [ ]:
IMG_SIZE = cfg.img_size

# Warmup
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device="cuda")
for _ in range(10):
    with torch.no_grad(), torch.amp.autocast("cuda"):
        _ = model(dummy)
torch.cuda.synchronize()
print("Warmup complete.")

monitor = GPUMonitor(interval_ms=100)
frame_times = []

torch.cuda.synchronize()
monitor.start()
t_start = time.time()
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    t0 = time.time()

    img = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(img.astype(np.float32) / 255.0)
    tensor = tensor.permute(2, 0, 1).unsqueeze(0).to("cuda")

    with torch.no_grad(), torch.amp.autocast("cuda"):
        _ = model(tensor)

    torch.cuda.synchronize()
    frame_times.append((time.time() - t0) * 1000)
    frame_idx += 1
    if frame_idx % 200 == 0:
        print(f"  Frame {frame_idx}/{total_frames}")

torch.cuda.synchronize()
t_end = time.time()
monitor.stop()
cap.release()

total_time = t_end - t_start
avg_fps = frame_idx / total_time
avg_lat = np.mean(frame_times)
p95_lat = np.percentile(frame_times, 95)
p99_lat = np.percentile(frame_times, 99)

print(f"\nDone: {frame_idx} frames in {total_time:.1f}s")
print(f"  Avg FPS:     {avg_fps:.1f}")
print(f"  Avg latency: {avg_lat:.2f}ms")
print(f"  P95 latency: {p95_lat:.2f}ms")
print(f"  P99 latency: {p99_lat:.2f}ms")

## 6 — Per-Second Report

In [ ]:
report = monitor.per_second_report(t_start)

# Save CSV
csv_path = os.path.join(OUTPUT_DIR, "gpu_per_second.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["second", "gpu_util_avg", "gpu_util_max", "mem_used_avg_mb"])
    w.writeheader()
    w.writerows(report)
print(f"Report saved: {csv_path}")

# Save latencies
lat_path = os.path.join(OUTPUT_DIR, "frame_latencies.csv")
with open(lat_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["frame", "latency_ms"])
    for i, t in enumerate(frame_times):
        w.writerow([i, f"{t:.3f}"])

## 7 — Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

seconds = [r["second"] for r in report]
gpu_avg = [r["gpu_util_avg"] for r in report]
gpu_max = [r["gpu_util_max"] for r in report]
mem_gb = [r["mem_used_avg_mb"] / 1024 for r in report]

axes[0,0].fill_between(seconds, gpu_avg, alpha=0.3, color="blue")
axes[0,0].plot(seconds, gpu_avg, label="Avg", color="blue")
axes[0,0].plot(seconds, gpu_max, label="Max", color="red", alpha=0.5)
axes[0,0].set_title("GPU Utilization per Second")
axes[0,0].set_ylabel("GPU %")
axes[0,0].legend()
axes[0,0].set_ylim(0, 105)

axes[0,1].plot(seconds, mem_gb, color="green")
axes[0,1].set_title("GPU Memory Usage")
axes[0,1].set_ylabel("GB")

axes[1,0].hist(frame_times, bins=50, color="orange", edgecolor="black", alpha=0.7)
axes[1,0].axvline(avg_lat, color="red", linestyle="--", label=f"Avg: {avg_lat:.1f}ms")
axes[1,0].axvline(p95_lat, color="purple", linestyle="--", label=f"P95: {p95_lat:.1f}ms")
axes[1,0].set_title("Frame Latency Distribution")
axes[1,0].set_xlabel("Latency (ms)")
axes[1,0].legend()

window = 30
if len(frame_times) > window:
    rolling = [1000.0 / np.mean(frame_times[max(0,i-window):i+1]) for i in range(len(frame_times))]
    axes[1,1].plot(rolling, linewidth=0.5, color="teal")
    axes[1,1].axhline(avg_fps, color="red", linestyle="--", label=f"Avg: {avg_fps:.0f} FPS")
    axes[1,1].set_title(f"Rolling FPS (window={window})")
    axes[1,1].set_ylabel("FPS")
    axes[1,1].legend()

plt.suptitle(f"GPU Profiling — {gpu_name}", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "gpu_profile.png"), dpi=150)
plt.show()

In [ ]:
# Copy to Drive
import shutil

drive_out = os.path.join(ECOCAR_ROOT, "profiling")
os.makedirs(drive_out, exist_ok=True)
for f in ["gpu_per_second.csv", "frame_latencies.csv", "gpu_profile.png"]:
    src = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)

print(f"\n{'='*55}")
print(f"  GPU PROFILING SUMMARY")
print(f"{'='*55}")
print(f"  GPU:          {gpu_name}")
print(f"  Frames:       {frame_idx}")
print(f"  Avg FPS:      {avg_fps:.1f}")
print(f"  Avg latency:  {avg_lat:.2f}ms")
print(f"  P95 latency:  {p95_lat:.2f}ms")
print(f"  Avg GPU util: {np.mean(gpu_avg):.1f}%")
print(f"  Avg GPU mem:  {np.mean(mem_gb):.2f} GB")
print(f"  Results:      {drive_out}")
print(f"{'='*55}")